# Project 8 — Module 9: Fundamentos de Big Data
## Lesson 04 — Modeling

| | |
|---|---|
| **Author** | Jose Marcel Lopez Pino |
| **Framework** | CRISP-DM + LEAN |
| **Phase** | 04 — Modeling |
| **Module** | M9 — Fundamentos de Big Data (Alkemy Bootcamp) |
| **Dataset** | Brazilian E-Commerce (Olist) + Synthetic clickstream |
| **Date** | 2026-04 |

---

> **Executive Summary:**
> This notebook covers two consigna lessons: L4 (Spark SQL + DataFrames) and
> L5 (MLlib pipeline). It transforms raw DataFrames with explicit schemas,
> executes Spark SQL queries to generate business metrics (revenue by category,
> top products, customer value), saves results in Parquet format, and builds
> an MLlib pipeline with VectorAssembler + StringIndexer for Logistic Regression
> (customer classification) and K-Means (customer segmentation). The key output
> is a complete ML pipeline with trained models and Parquet artifacts.

## Table of Contents

1. [Environment Setup — PySpark + Data Reload](#1-environment-setup)
2. [DataFrames with Explicit Schemas](#2-dataframes-with-explicit-schemas)
3. [Spark SQL — Business Metrics](#3-spark-sql--business-metrics)
4. [Save Results in Parquet Format](#4-save-results-in-parquet-format)
5. [Feature Engineering for MLlib](#5-feature-engineering-for-mllib)
6. [MLlib — Logistic Regression (Supervised)](#6-mllib--logistic-regression)
7. [MLlib — K-Means Clustering (Unsupervised)](#7-mllib--k-means-clustering)
8. [LEAN Filter — Waste Elimination Review](#8-lean-filter)
9. [Decisions Log — Lesson 04](#9-decisions-log)
10. [Next Steps — Lesson 05 Preview](#10-next-steps)

---

## 1. Environment Setup — PySpark + Data Reload <a id='1-environment-setup'></a>

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
import os
import sys
import warnings
from pathlib import Path

os.environ['HADOOP_HOME'] = str(Path.home() / 'hadoop')

os.environ['PYSPARK_PYTHON'] = sys.executable

import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType
)

# MLlib
from pyspark.ml.feature import VectorAssembler, StringIndexer, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
    ClusteringEvaluator
)
from pyspark.ml import Pipeline

import matplotlib.pyplot as plt
import seaborn as sns

warnings.formatwarning = lambda msg, *args, **kwargs: f'Warning: {msg}\n'
warnings.simplefilter('always')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Blues_d')

DATA_RAW       = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_FINAL     = Path('../data/final')

print('Imports ready.')

In [ ]:
# =============================================================================
# SPARK SESSION + DATA RELOAD
# =============================================================================
spark = (
    SparkSession.builder
    .appName('RetailMax-BigData-Pipeline')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel('WARN')

# Reload base DataFrames
olist_files = {
    'orders':       str(DATA_RAW / 'olist_orders_dataset.csv'),
    'order_items':  str(DATA_RAW / 'olist_order_items_dataset.csv'),
    'payments':     str(DATA_RAW / 'olist_order_payments_dataset.csv'),
    'reviews':      str(DATA_RAW / 'olist_order_reviews_dataset.csv'),
    'products':     str(DATA_RAW / 'olist_products_dataset.csv'),
    'customers':    str(DATA_RAW / 'olist_customers_dataset.csv'),
    'sellers':      str(DATA_RAW / 'olist_sellers_dataset.csv'),
    'geolocation':  str(DATA_RAW / 'olist_geolocation_dataset.csv'),
    'categories':   str(DATA_RAW / 'product_category_name_translation.csv'),
}

dfs = {}
for name, path in olist_files.items():
    dfs[name] = spark.read.csv(path, header=True, inferSchema=True, encoding='utf-8')

print(f'Spark {spark.version} | {len(dfs)} DataFrames loaded.')

---

## 2. DataFrames with Explicit Schemas <a id='2-dataframes-with-explicit-schemas'></a>

Apply explicit `StructType` schemas to enforce correct data types.
This replaces `inferSchema=True` which mistyped `review_score` as string
(discovered in NB 03). Explicit schemas are a production best practice —
they prevent silent type errors in downstream transformations.

In [ ]:
# =============================================================================
# EXPLICIT SCHEMAS — production best practice
# =============================================================================

# Reviews — fix review_score from string to int
reviews_schema = StructType([
    StructField('review_id', StringType(), True),
    StructField('order_id', StringType(), True),
    StructField('review_score', IntegerType(), True),
    StructField('review_comment_title', StringType(), True),
    StructField('review_comment_message', StringType(), True),
    StructField('review_creation_date', StringType(), True),
    StructField('review_answer_timestamp', StringType(), True),
])

# Reload reviews with explicit schema
dfs['reviews'] = spark.read.csv(
    str(DATA_RAW / 'olist_order_reviews_dataset.csv'),
    header=True,
    schema=reviews_schema,
    encoding='utf-8',
    mode='DROPMALFORMED'  # Drop rows that don't match schema
)

# Verify fix
print('Reviews schema (explicit):')
dfs['reviews'].printSchema()

reviews_count = dfs['reviews'].count()
valid_scores = dfs['reviews'].filter(F.col('review_score').isNotNull()).count()
print(f'Total reviews: {reviews_count:,} | Valid scores: {valid_scores:,} | Dropped: {reviews_count - valid_scores:,}')

---

## 3. Spark SQL — Business Metrics <a id='3-spark-sql--business-metrics'></a>

Register DataFrames as temporary SQL views and execute queries to generate
business metrics required by the consigna: revenue by category, top products,
and customer value analysis.

In [ ]:
# =============================================================================
# REGISTER TEMP VIEWS FOR SPARK SQL
# =============================================================================
dfs['orders'].createOrReplaceTempView('orders')
dfs['order_items'].createOrReplaceTempView('order_items')
dfs['payments'].createOrReplaceTempView('payments')
dfs['reviews'].createOrReplaceTempView('reviews')
dfs['products'].createOrReplaceTempView('products')
dfs['customers'].createOrReplaceTempView('customers')
dfs['categories'].createOrReplaceTempView('categories')

print('Temp views registered:')
for view in spark.catalog.listTables():
    if view.isTemporary:
        print(f'  {view.name}')

In [ ]:
# =============================================================================
# SPARK SQL: Revenue by product category (Top 15)
# =============================================================================
revenue_by_category = spark.sql("""
    SELECT
        COALESCE(c.product_category_name_english, 'unknown') AS category,
        COUNT(DISTINCT oi.order_id)                          AS total_orders,
        SUM(oi.price + oi.freight_value)                     AS total_revenue,
        AVG(oi.price + oi.freight_value)                     AS avg_ticket,
        COUNT(oi.order_item_id)                              AS items_sold
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    LEFT JOIN categories c ON p.product_category_name = c.product_category_name
    GROUP BY COALESCE(c.product_category_name_english, 'unknown')
    ORDER BY total_revenue DESC
    LIMIT 15
""")

print('SPARK SQL: Revenue by Category — Top 15')
revenue_by_category.show(15, truncate=False)

In [ ]:
# =============================================================================
# SPARK SQL: Top 10 products by revenue
# =============================================================================
top_products = spark.sql("""
    SELECT
        oi.product_id,
        COALESCE(c.product_category_name_english, 'unknown') AS category,
        COUNT(oi.order_item_id)              AS times_sold,
        SUM(oi.price)                        AS total_revenue,
        AVG(oi.price)                        AS avg_price
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    LEFT JOIN categories c ON p.product_category_name = c.product_category_name
    GROUP BY oi.product_id, COALESCE(c.product_category_name_english, 'unknown')
    ORDER BY total_revenue DESC
    LIMIT 10
""")

print('SPARK SQL: Top 10 Products by Revenue')
top_products.show(10, truncate=30)

In [ ]:
# =============================================================================
# SPARK SQL: Customer value features (for ML pipeline)
# =============================================================================
customer_features = spark.sql("""
    SELECT
        cu.customer_unique_id,
        cu.customer_state,
        COUNT(DISTINCT o.order_id)                       AS order_count,
        SUM(oi.price + oi.freight_value)                 AS total_spent,
        AVG(oi.price + oi.freight_value)                 AS avg_ticket,
        COUNT(oi.order_item_id)                          AS items_purchased,
        AVG(r.review_score)                              AS avg_review_score,
        DATEDIFF(
            MAX(o.order_purchase_timestamp),
            MIN(o.order_purchase_timestamp)
        )                                                AS customer_lifespan_days
    FROM customers cu
    JOIN orders o ON cu.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    LEFT JOIN reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY cu.customer_unique_id, cu.customer_state
    HAVING COUNT(DISTINCT o.order_id) >= 1
""")

print(f'Customer features: {customer_features.count():,} customers')
customer_features.show(10, truncate=False)

In [ ]:
# =============================================================================
# SPARK SQL: Payment method analysis
# =============================================================================
payment_analysis = spark.sql("""
    SELECT
        payment_type,
        COUNT(*)                AS transaction_count,
        SUM(payment_value)      AS total_value,
        AVG(payment_value)      AS avg_value,
        AVG(payment_installments) AS avg_installments
    FROM payments
    GROUP BY payment_type
    ORDER BY total_value DESC
""")

print('SPARK SQL: Payment Method Analysis')
payment_analysis.show(truncate=False)

---

## 4. Save Results in Parquet Format <a id='4-save-results-in-parquet-format'></a>

Parquet is a columnar storage format optimized for analytical queries:
- **Compression:** 50–70% smaller than CSV
- **Schema enforcement:** Types preserved, no re-inference needed
- **Predicate pushdown:** Spark reads only the columns/rows needed

> **Analogy (ICI):** Parquet is like organizing inventory by SKU category
> instead of arrival order — retrieval is faster because you go directly
> to the relevant section.

In [ ]:
# =============================================================================
# SAVE RESULTS IN PARQUET FORMAT
# =============================================================================
# Customer features — main input for ML pipeline
customer_features.write.parquet(str(DATA_PROCESSED / 'customer_features.parquet'),mode='overwrite')

# Revenue by category — business metric
revenue_by_category.write.parquet(
    str(DATA_PROCESSED / 'revenue_by_category.parquet'),
    mode='overwrite'
)

# Top products — business metric
top_products.write.parquet(
    str(DATA_PROCESSED / 'top_products.parquet'),
    mode='overwrite'
)

# Payment analysis — business metric
payment_analysis.write.parquet(
    str(DATA_PROCESSED / 'payment_analysis.parquet'),
    mode='overwrite'
)

print('Parquet files saved to data/processed/')
print()

# Verify Parquet read-back
cf_parquet = spark.read.parquet(str(DATA_PROCESSED / 'customer_features.parquet'))
print(f'Parquet read-back: customer_features — {cf_parquet.count():,} rows')
cf_parquet.printSchema()

In [ ]:
try:
    customer_features.write.parquet(
        str(DATA_PROCESSED / 'customer_features.parquet'),
        mode='overwrite'
    )
except Exception as e:
    print(str(e)[:500])

In [ ]:
try:
    customer_features.write.parquet(
        str(DATA_PROCESSED / 'customer_features.parquet'),
        mode='overwrite'
    )
except Exception as e:
    print(str(e)[-1000:])

In [ ]:
# Test simple write
test_df = spark.createDataFrame([(1, "a"), (2, "b")], ["id", "val"])
try:
    test_df.write.parquet(str(DATA_PROCESSED / 'test.parquet'), mode='overwrite')
    print("Parquet write: OK")
except Exception as e:
    # Try CSV as fallback
    print(f"Parquet FAILED: {str(e)[:200]}")
    print("\nTrying CSV instead...")
    try:
        test_df.write.csv(str(DATA_PROCESSED / 'test_csv'), header=True, mode='overwrite')
        print("CSV write: OK")
    except Exception as e2:
        print(f"CSV also FAILED: {str(e2)[:200]}")

---

## 5. Feature Engineering for MLlib <a id='5-feature-engineering-for-mllib'></a>

Prepare features for the ML pipeline:
- **Target variable (supervised):** Binary label — High Value (1) vs Low Value (0)
  based on total_spent above/below median
- **Features:** order_count, total_spent, avg_ticket, items_purchased,
  avg_review_score, customer_lifespan_days
- **Categorical encoding:** customer_state via StringIndexer

In [ ]:
# =============================================================================
# FEATURE ENGINEERING
# =============================================================================
# Load from Parquet (demonstrates Parquet → ML flow)
cf = spark.read.parquet(str(DATA_PROCESSED / 'customer_features.parquet'))

# Handle nulls in review score (not all customers left reviews)
cf = cf.fillna({'avg_review_score': 0.0, 'customer_lifespan_days': 0})

# Create binary label: High Value = total_spent > median
median_spent = cf.approxQuantile('total_spent', [0.5], 0.01)[0]
print(f'Median total_spent: {median_spent:.2f}')

cf = cf.withColumn(
    'high_value',
    F.when(F.col('total_spent') > median_spent, 1).otherwise(0)
)

# Check label distribution
print('\nLabel distribution:')
cf.groupBy('high_value').agg(
    F.count('*').alias('count')
).show()

# Feature columns
feature_cols = [
    'order_count', 'total_spent', 'avg_ticket',
    'items_purchased', 'avg_review_score', 'customer_lifespan_days'
]

print(f'Feature columns: {feature_cols}')
print(f'Target: high_value (binary)')
print(f'Customers: {cf.count():,}')

In [ ]:
# =============================================================================
# MLLIB PIPELINE COMPONENTS
# =============================================================================
# StringIndexer for categorical state
state_indexer = StringIndexer(
    inputCol='customer_state',
    outputCol='state_index',
    handleInvalid='keep'
)

# Feature columns including indexed state
all_feature_cols = feature_cols + ['state_index']

# VectorAssembler — combines all features into a single vector column
assembler = VectorAssembler(
    inputCols=all_feature_cols,
    outputCol='features_raw',
    handleInvalid='skip'
)

# StandardScaler — normalize features for better model convergence
scaler = StandardScaler(
    inputCol='features_raw',
    outputCol='features',
    withStd=True,
    withMean=True
)

print('Pipeline components defined:')
print(f'  1. StringIndexer: customer_state → state_index')
print(f'  2. VectorAssembler: {len(all_feature_cols)} features → features_raw')
print(f'  3. StandardScaler: features_raw → features')

---

## 6. MLlib — Logistic Regression (Supervised) <a id='6-mllib--logistic-regression'></a>

Classify customers as High Value (1) vs Low Value (0) using Logistic Regression.
This is the supervised model required by the consigna.

In [ ]:
# =============================================================================
# LOGISTIC REGRESSION — Train/Test Split
# =============================================================================
# Split data
train_df, test_df = cf.randomSplit([0.8, 0.2], seed=42)
print(f'Train: {train_df.count():,} | Test: {test_df.count():,}')

# Define Logistic Regression
lr = LogisticRegression(
    featuresCol='features',
    labelCol='high_value',
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0  # L2 regularization
)

# Build pipeline: indexer → assembler → scaler → logistic regression
lr_pipeline = Pipeline(stages=[state_indexer, assembler, scaler, lr])

# Train
print('Training Logistic Regression pipeline...')
lr_model = lr_pipeline.fit(train_df)
print('Training complete.')

In [ ]:
# =============================================================================
# LOGISTIC REGRESSION — Predictions & Evaluation
# =============================================================================
lr_predictions = lr_model.transform(test_df)

# Show sample predictions
print('Sample predictions:')
lr_predictions.select(
    'customer_unique_id', 'total_spent', 'order_count',
    'high_value', 'prediction', 'probability'
).show(10, truncate=30)

# Evaluation metrics
# AUC-ROC
auc_evaluator = BinaryClassificationEvaluator(
    labelCol='high_value',
    metricName='areaUnderROC'
)
auc_roc = auc_evaluator.evaluate(lr_predictions)

# Accuracy, Precision, Recall, F1
acc_evaluator = MulticlassClassificationEvaluator(
    labelCol='high_value',
    predictionCol='prediction'
)

accuracy  = acc_evaluator.evaluate(lr_predictions, {acc_evaluator.metricName: 'accuracy'})
precision = acc_evaluator.evaluate(lr_predictions, {acc_evaluator.metricName: 'weightedPrecision'})
recall    = acc_evaluator.evaluate(lr_predictions, {acc_evaluator.metricName: 'weightedRecall'})
f1        = acc_evaluator.evaluate(lr_predictions, {acc_evaluator.metricName: 'f1'})

print('Logistic Regression — Test Metrics:')
print(f'  AUC-ROC    : {auc_roc:.4f}')
print(f'  Accuracy   : {accuracy:.4f}')
print(f'  Precision  : {precision:.4f}')
print(f'  Recall     : {recall:.4f}')
print(f'  F1 Score   : {f1:.4f}')

In [ ]:
# =============================================================================
# CONFUSION MATRIX — Logistic Regression
# =============================================================================
confusion = (
    lr_predictions
    .groupBy('high_value', 'prediction')
    .agg(F.count('*').alias('count'))
    .orderBy('high_value', 'prediction')
)

print('Confusion Matrix:')
confusion.show()

# Visualize with matplotlib
conf_pd = confusion.toPandas()
conf_matrix = conf_pd.pivot(
    index='high_value', columns='prediction', values='count'
).fillna(0).astype(int)

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Logistic Regression — Confusion Matrix')
plt.tight_layout()
plt.show()

---

## 7. MLlib — K-Means Clustering (Unsupervised) <a id='7-mllib--k-means-clustering'></a>

Segment customers into groups for targeted marketing using K-Means.
This is the unsupervised model required by the consigna.

In [ ]:
# =============================================================================
# K-MEANS — Feature preparation (reuse pipeline components)
# =============================================================================
# Build feature pipeline (same indexer + assembler + scaler, no classifier)
feature_pipeline = Pipeline(stages=[state_indexer, assembler, scaler])
feature_model = feature_pipeline.fit(cf)
cf_features = feature_model.transform(cf)

print(f'Features prepared for clustering: {cf_features.count():,} customers')
print(f'Feature vector size: {len(all_feature_cols)}')

In [ ]:
# =============================================================================
# K-MEANS — Elbow Method (find optimal K)
# =============================================================================
silhouette_scores = []
cost_values = []
k_range = range(2, 8)

evaluator = ClusteringEvaluator(
    featuresCol='features',
    metricName='silhouette'
)

for k in k_range:
    kmeans = KMeans(
        featuresCol='features',
        k=k,
        seed=42,
        maxIter=50
    )
    model = kmeans.fit(cf_features)
    predictions = model.transform(cf_features)

    silhouette = evaluator.evaluate(predictions)
    cost = model.summary.trainingCost

    silhouette_scores.append(silhouette)
    cost_values.append(cost)
    print(f'  K={k}: Silhouette={silhouette:.4f} | Cost={cost:,.0f}')

# Plot elbow
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(list(k_range), cost_values, 'bo-')
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Training Cost (WSSSE)')
ax1.set_title('Elbow Method')

ax2.plot(list(k_range), silhouette_scores, 'ro-')
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Analysis')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# K-MEANS — Final model (K=4 — typical for customer segmentation)
# =============================================================================
# Select K based on elbow + silhouette analysis
# Adjust K if the elbow/silhouette plots suggest a different value
OPTIMAL_K = 4

kmeans_final = KMeans(
    featuresCol='features',
    k=OPTIMAL_K,
    seed=42,
    maxIter=100
)

kmeans_model = kmeans_final.fit(cf_features)
km_predictions = kmeans_model.transform(cf_features)

# Silhouette score
final_silhouette = evaluator.evaluate(km_predictions)
print(f'K-Means (K={OPTIMAL_K}) — Silhouette Score: {final_silhouette:.4f}')
print(f'Training Cost (WSSSE): {kmeans_model.summary.trainingCost:,.0f}')

# Cluster distribution
print('\nCluster Distribution:')
km_predictions.groupBy('prediction').agg(
    F.count('*').alias('customers')
).orderBy('prediction').show()

In [ ]:
# =============================================================================
# K-MEANS — Cluster Profiles (marketing insights)
# =============================================================================
cluster_profiles = (
    km_predictions
    .groupBy('prediction')
    .agg(
        F.count('*').alias('customers'),
        F.round(F.avg('order_count'), 2).alias('avg_orders'),
        F.round(F.avg('total_spent'), 2).alias('avg_spent'),
        F.round(F.avg('avg_ticket'), 2).alias('avg_ticket'),
        F.round(F.avg('items_purchased'), 2).alias('avg_items'),
        F.round(F.avg('avg_review_score'), 2).alias('avg_review'),
        F.round(F.avg('customer_lifespan_days'), 0).alias('avg_lifespan_days')
    )
    .orderBy('prediction')
)

print('Cluster Profiles — Marketing Insights:')
cluster_profiles.show(truncate=False)

# Visualize cluster profiles
profiles_pd = cluster_profiles.toPandas()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(profiles_pd['prediction'].astype(str), profiles_pd['avg_spent'])
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Avg Spent (BRL)')
axes[0].set_title('Average Spending by Cluster')

axes[1].bar(profiles_pd['prediction'].astype(str), profiles_pd['avg_orders'])
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Avg Orders')
axes[1].set_title('Average Orders by Cluster')

axes[2].bar(profiles_pd['prediction'].astype(str), profiles_pd['avg_review'])
axes[2].set_xlabel('Cluster')
axes[2].set_ylabel('Avg Review Score')
axes[2].set_title('Average Review by Cluster')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# SAVE ML RESULTS TO PARQUET
# =============================================================================
# LogReg predictions
lr_predictions.select(
    'customer_unique_id', 'customer_state', 'order_count',
    'total_spent', 'avg_ticket', 'items_purchased',
    'avg_review_score', 'high_value', 'prediction'
).write.parquet(
    str(DATA_FINAL / 'lr_predictions.parquet'),
    mode='overwrite'
)

# K-Means predictions
km_predictions.select(
    'customer_unique_id', 'customer_state', 'order_count',
    'total_spent', 'avg_ticket', 'items_purchased',
    'avg_review_score', 'prediction'
).withColumnRenamed('prediction', 'cluster').write.parquet(
    str(DATA_FINAL / 'km_clusters.parquet'),
    mode='overwrite'
)

# Cluster profiles
cluster_profiles.write.parquet(
    str(DATA_FINAL / 'cluster_profiles.parquet'),
    mode='overwrite'
)

print('ML results saved to data/final/')
print('  - lr_predictions.parquet')
print('  - km_clusters.parquet')
print('  - cluster_profiles.parquet')

---

## 8. LEAN Filter — Waste Elimination Review <a id='8-lean-filter'></a>

| LEAN Question | Answer | Action |
|---|---|---|
| Does every SQL query produce a business metric? | ✅ Revenue by category, top products, customer value, payment analysis — all actionable | Proceed |
| Are both ML models justified? | ✅ LogReg = classification (consigna), K-Means = segmentation (consigna) — both required | Proceed |
| Is Parquet justified over CSV? | ✅ Schema enforcement + compression + predicate pushdown — demonstrated with read-back | Proceed |
| Could we use fewer features? | ⚠️ 7 features is already minimal; removing any loses business signal | Keep current set |

---

## 9. Decisions Log — Lesson 04 <a id='9-decisions-log'></a>

| # | Decision | Rationale | Alternatives Considered | LEAN Value? |
|---|----------|-----------|------------------------|-------------|
| D1 | Explicit schema for reviews (DROPMALFORMED) | Fixes review_score string issue from NB 03; drops 4,937 malformed rows cleanly | try_cast in SQL (works but adds complexity), keep as string (breaks ML) | ✅ |
| D2 | Median split for High/Low Value label | Creates balanced classes; median is robust to outliers | Arbitrary threshold (biased), percentile-based (same concept, more complex) | ✅ |
| D3 | StandardScaler before LogReg | LogReg is sensitive to feature scale; scaler normalizes all features | No scaling (poor convergence), MinMaxScaler (sensitive to outliers) | ✅ |
| D4 | K=4 as default, adjustable after elbow analysis | 4 segments is standard in retail (VIP, regular, at-risk, dormant) | K=3 (too coarse), K=6 (too granular for marketing) | ✅ |
| D5 | Combined L4+L5 in single notebook | Spark SQL is the feature engineering that feeds MLlib — splitting them breaks the analytical flow | Separate notebooks (artificial boundary between SQL and ML) | ✅ |

---

## 10. Next Steps — Lesson 05 Preview <a id='10-next-steps'></a>

| Priority | Next Step | Target |
|---|---|---|
| 🔴 Immediate | Deep evaluation of LogReg and K-Means results | NB 05 |
| 🔴 Immediate | Interpret cluster profiles and generate marketing recommendations | NB 05 |
| 🟡 Short-term | Assess model against success metrics defined in NB 01 (AUC > 0.75, Silhouette > 0.40) | NB 05 |
| 🟢 Long-term | Compile final report and consigna compliance check | NB 06 |

---

**← Previous:** [03 — Data Preparation](./03_data_preparation.ipynb)
**Next Phase →** [05 — Evaluation](./05_evaluation.ipynb)

In [ ]:
# =============================================================================
# NB04 — FINAL STATUS
# (Delete before final commit)
# =============================================================================
print('=' * 65)
print('NOTEBOOK 04 — FINAL STATUS')
print('=' * 65)

# Parquet files saved
import os
print('\n--- PARQUET FILES SAVED ---')
for folder in [DATA_PROCESSED, DATA_FINAL]:
    for item in sorted(folder.iterdir()):
        if item.suffix == '.parquet' or item.is_dir():
            print(f'  {item}')

# Logistic Regression results
print('\n--- LOGISTIC REGRESSION ---')
lr_predictions = spark.read.parquet(str(DATA_FINAL / 'lr_predictions.parquet'))
print(f'Rows: {lr_predictions.count():,}')
lr_predictions.groupBy('high_value', 'prediction').count().orderBy('high_value', 'prediction').show()

# K-Means results
print('\n--- K-MEANS CLUSTERING ---')
km_clusters = spark.read.parquet(str(DATA_FINAL / 'km_clusters.parquet'))
print(f'Rows: {km_clusters.count():,}')
km_clusters.groupBy('cluster').count().orderBy('cluster').show()

# Cluster profiles
print('\n--- CLUSTER PROFILES ---')
cluster_profiles = spark.read.parquet(str(DATA_FINAL / 'cluster_profiles.parquet'))
cluster_profiles.show(truncate=False)

# Columns available
print('\n--- LR_PREDICTIONS COLUMNS ---')
print(lr_predictions.columns)
print('\n--- KM_CLUSTERS COLUMNS ---')
print(km_clusters.columns)

print('\n' + '=' * 65)
print('Ready for Notebook 05.')
print('=' * 65)

In [ ]:
km_clusters = spark.read.parquet(str(DATA_FINAL / 'km_clusters.parquet'))
km_clusters.groupBy('cluster').count().orderBy('cluster').show()

print('---')
cluster_profiles = spark.read.parquet(str(DATA_FINAL / 'cluster_profiles.parquet'))
cluster_profiles.show(truncate=False)

print('---')
print('KM columns:', km_clusters.columns)
print('LR columns:', spark.read.parquet(str(DATA_FINAL / 'lr_predictions.parquet')).columns)